In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Loading dataset
df = pd.read_csv('cclass_data.csv')

# Dropping the completely empty rows
df = df.dropna(how='all')

# Dropping the unecessary columns of model and reference 
df = df.drop(columns=['model', 'reference'])

df.head()

,year,price,transmission,mileage,fuel type,engine size,mileage2,fuel type2,engine size2
0,2020.0,"£30,495",Automatic,NaN,Diesel,2,"1,200",NaN,NaN
1,2020.0,"£29,989",Automatic,NaN,Petrol,1.5,"1,000",NaN,NaN
2,2020.0,"£37,899",Automatic,NaN,Diesel,2,500,NaN,NaN
3,2019.0,"£30,399",Automatic,NaN,Diesel,2,"5,000",NaN,NaN
4,2019.0,"£29,899",Automatic,NaN,Diesel,2,"4,500",NaN,NaN


In [2]:
df.tail()

,year,price,transmission,mileage,fuel type,engine size,mileage2,fuel type2,engine size2
4001,2017.0,"£14,700",Manual,"31,357",25,£150,70.6,Diesel,1.598
4002,2018.0,"£18,500",Automatic,"28,248",31,£150,64.2,Diesel,2.143
4003,2014.0,"£11,900",Manual,"48,055",31,£20,65.7,Diesel,2.143
4004,2014.0,"£11,300",Automatic,"49,865",46,£145,56.5,Diesel,2.143
4005,2014.0,"£14,800",Automatic,"55,445",37,£30,64.2,Diesel,2.143


In [3]:
# Fixing the data misalignment issue where earlier rows use  “mileage”, “fuel_type”, and “engine_size”
# but later rows change to "mileage2", "fuel_type2", and "engine_size2" This is the same data, but misaligned.

# Spliting the dataset into 2 groups for migrating
df_newer = df[df['mileage'].isnull()].copy()
df_shifted = df[df['mileage'].notnull()].copy()

# Dropping the unneeded original columns for mileage and fuel type
df_newer = df_newer.drop(columns=['mileage'])
df_shifted = df_shifted.drop(columns=['fuel type'])

# Fixing the shifted data by renaming them to the correct name
df_shifted = df_shifted.rename(columns={
    'engine size': 'tax',         
    'mileage2': 'mpg',            
    'fuel type2': 'fuel type',    
    'engine size2': 'engine size' })
df_newer = df_newer.rename(columns={
    'mileage2': 'mileage' })
df_newer['tax'] = np.nan
df_newer['mpg'] = np.nan

# Combining the cleaned datasets
df_clean = pd.concat([df_shifted, df_newer], ignore_index=True)

# Dropping any duplicate columns
cols_to_drop = ['engine size2', 'fuel type2']
df_clean = df_clean.drop(columns=cols_to_drop, errors='ignore')

df_clean.head()


,year,price,transmission,mileage,tax,mpg,fuel type,engine size
0,2013.0,"£9,995",Automatic,"44,900",£160,46.3,Petrol,1.6
1,2012.0,"£6,995",Automatic,"88,200",£125,58.9,Diesel,2.1
2,2012.0,"£7,495",Automatic,"115,000",£145,54.3,Diesel,2.1
3,2011.0,"£8,995",Automatic,"69,250",£150,53.3,Diesel,2.1
4,2015.0,"£14,995",Automatic,"49,850",£30,62.8,Diesel,2.1


In [4]:
#Cleaning up symbols and strings

# Clean up symbols like commas and currency in the price, tax, mileage columns
df_clean['price'] = df_clean['price'].astype(str).str.replace('£', '').str.replace(',', '')
df_clean['tax'] = df_clean['tax'].astype(str).str.replace('£', '').str.replace(',', '')
df_clean['mileage'] = df_clean['mileage'].astype(str).str.replace(',', '')

# Convert non-numeric type to numeric
df_clean['price'] = pd.to_numeric(df_clean['price'], errors='coerce')
df_clean['tax'] = pd.to_numeric(df_clean['tax'], errors='coerce')
df_clean['mileage'] = pd.to_numeric(df_clean['mileage'], errors='coerce')
df_clean['engine size'] = pd.to_numeric(df_clean['engine size'], errors='coerce')

# Convert the mpg to numeric so median mpg can be calculated
df_clean['mpg'] = pd.to_numeric(df_clean['mpg'], errors='coerce')

# Converting engine size into liters
# Formula: divide the volume value by 1000
df_clean.loc[df_clean['engine size'] > 10, 'engine size'] = df_clean['engine size'] / 1000

# Convert GBP to CAD (Using exchange rate of 1.84 CAD per GBP on March 31/2026)
df_clean['price_cad'] = df_clean['price'] * 1.84
df_clean = df_clean.drop(columns=['price']) # Drop the original pound column

In [8]:
# Dealing with Missing Entries

# Numerical Data
# Median value to replace numerical attribute rows with missing data
median_tax = df_clean['tax'].median()
median_mpg = df_clean['mpg'].median()

# Using .fillna to replace empty rows (numerical data) with respective median
df_clean['tax'] = df_clean['tax'].fillna(median_tax)
df_clean['mpg'] = df_clean['mpg'].fillna(median_mpg)

# Categorial Data
# Find the most common fuel type
most_common_fuel = df_clean['fuel type'].mode()[0]

# Using .fillna to replace empty rows with most common fuel type
df_clean['fuel type'] = df_clean['fuel type'].fillna(most_common_fuel)

# Dropping rows with unknown values
df_clean = df_clean.dropna()

df_clean.head()

,transmission,mileage,tax,mpg,fuel type,engine size,price_cad,car_age
0,Automatic,44900.0,160.0,46.3,Petrol,1.6,18390.8,13.0
1,Automatic,88200.0,125.0,58.9,Diesel,2.1,12870.8,14.0
2,Automatic,115000.0,145.0,54.3,Diesel,2.1,13790.8,14.0
3,Automatic,69250.0,150.0,53.3,Diesel,2.1,16550.8,15.0
4,Automatic,49850.0,30.0,62.8,Diesel,2.1,27590.8,11.0


In [9]:
#Exporting cleaned dataset back to a new csv filed named cleaned_data.csv
#https://www.linkedin.com/pulse/cleaning-data-csv-file-using-python-sheetal-dhande-dandge-phd-/
df_clean.to_csv('cleaned_data.csv', index=False)
